<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B08%5D%20-%20Ingenieria_de_Variables_II/%5B01%5D%20-%20Notebooks/E5_Texto_Sparse_con_TruncatedSVD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E5 · Texto sparse con TruncatedSVD - Ingeniería de Variables II


## Introducción

El texto es el caso clásico de "muchísimas columnas". Al vectorizar con **TF-IDF**, creamos
**una columna por palabra**: miles de columnas y casi todas a cero (una matriz **dispersa**).

Sobre datos dispersos no usamos PCA (que necesita centrar los datos y rompería la
dispersión), sino **TruncatedSVD**, que reduce dimensiones funcionando directamente sobre la
matriz sparse. A esta combinación (TF-IDF + TruncatedSVD) se la conoce como **LSA** (Latent Semantic Analysis).

En este ejercicio vectorizamos reseñas, reducimos a **50 componentes** y comparamos
**columnas, tiempo y rendimiento**.

## Objetivos del ejercicio

- Vectorizar texto con **TF-IDF** y ver cuántas columnas (dispersas) genera.
- Reducir con **TruncatedSVD** a 50 componentes.
- Comparar **nº de columnas, tiempo y AUC** con y sin reducción.
- Entender por qué en sparse usamos TruncatedSVD y no PCA.

## Descripción del dataset (reseñas sintéticas)

Generamos reseñas **sintéticas** en español, etiquetadas como positivas (1) o negativas (0).
La señal está en los **adjetivos** (excelente, pésimo...); el resto son palabras de relleno.
Para que no sea un problema trivial, añadimos algo de ruido (adjetivos contrarios y alguna
etiqueta cambiada).

### 1. Importar librerías necesarias

In [ ]:
import numpy as np
import pandas as pd
from time import perf_counter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

### 2. Generar las reseñas

In [ ]:
import numpy as np
import pandas as pd

def generar_resenas(n=2000, ruido_contrario=0.4, prob_flip=0.08, semilla=42):
    # Genera reseñas sinteticas en español (positivas / negativas). La señal vive en
    # los adjetivos; el resto son palabras de relleno (ruido). Al vectorizar con TF-IDF
    # salen muchisimas columnas, casi todas a cero (matriz dispersa). 'ruido_contrario'
    # y 'prob_flip' hacen el problema mas realista (no perfecto).
    rng = np.random.default_rng(semilla)
    adj_pos = ["excelente", "genial", "fantástico", "maravilloso", "recomendable", "increíble",
               "perfecto", "bueno", "estupendo", "encantado", "rápido", "cómodo", "bonito", "útil"]
    adj_neg = ["pésimo", "horrible", "decepcionante", "malo", "lento", "caro", "defectuoso",
               "incómodo", "aburrido", "frustrante", "feo", "inútil", "ruidoso", "flojo"]
    sustantivos = ["producto", "servicio", "envío", "precio", "calidad", "atención", "experiencia",
                   "pedido", "artículo", "soporte", "embalaje", "diseño", "material", "reparto"]
    conectores = ["además", "aunque", "la verdad", "en general", "sin duda", "para mí", "al final",
                  "francamente", "con el tiempo", "en mi opinión", "honestamente", "la primera vez"]
    relleno = ["ayer", "casa", "tienda", "caja", "color", "tamaño", "semana", "ciudad", "mañana",
               "tarde", "amigo", "familia", "compra", "uso", "detalle", "momento", "gente", "mundo",
               "cosa", "parte", "manera", "sitio", "punto", "tema", "idea", "grupo", "número", "caso"]

    textos, y = [], []
    for _ in range(n):
        positivo = rng.random() < 0.5
        adjs = adj_pos if positivo else adj_neg
        otros = adj_neg if positivo else adj_pos
        frases = []
        for _ in range(int(rng.integers(2, 5))):
            s = rng.choice(sustantivos)
            a = rng.choice(adjs)
            c = rng.choice(conectores)
            rell = " ".join(rng.choice(relleno, size=int(rng.integers(2, 6))))
            frase = f"{c} el {s} fue {a} {rell}"
            if rng.random() < ruido_contrario:    # mete un adjetivo del sentimiento contrario
                frase += f" pero el {rng.choice(sustantivos)} resultó {rng.choice(otros)}"
            frases.append(frase)
        etiqueta = int(positivo)
        if rng.random() < prob_flip:              # algo de ruido de etiqueta (realista)
            etiqueta = 1 - etiqueta
        textos.append(". ".join(frases) + ".")
        y.append(etiqueta)

    return pd.DataFrame({"resena": textos, "sentimiento": y})

In [ ]:
df = generar_resenas(n=2000, semilla=42)
print("Reseñas:", len(df), "| % positivas:", round(df["sentimiento"].mean(), 2))
print("\nEjemplo positivo:\n ", df[df.sentimiento == 1]["resena"].iloc[0][:160], "...")
print("\nEjemplo negativo:\n ", df[df.sentimiento == 0]["resena"].iloc[0][:160], "...")
df.head()

### 3. TF-IDF: muchas columnas y muy dispersas

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["resena"], df["sentimiento"], test_size=0.3, random_state=0, stratify=df["sentimiento"])

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2)   # palabras y pares de palabras
Xtr_tfidf = vec.fit_transform(X_train)                # fit SOLO con train
n_cols_tfidf = Xtr_tfidf.shape[1]

densidad = Xtr_tfidf.nnz / (Xtr_tfidf.shape[0] * Xtr_tfidf.shape[1])
print("Columnas TF-IDF:", n_cols_tfidf)

In [ ]:
print('Visualizando las primeras 5 filas de la matriz TF-IDF (convertida a densa para visualización):')

feature_names = vec.get_feature_names_out()

df_tfidf = pd.DataFrame(Xtr_tfidf[:5].toarray(), columns=feature_names)
display(df_tfidf.head())

print("Matriz dispersa: solo el", f"{densidad*100:.2f}%", "de las celdas son distintas de cero.")

Como puedes ver, la mayoría de los valores son cero, lo que confirma su naturaleza dispersa. Cada columna representa un término (palabra o par de palabras) y los valores son sus puntuaciones TF-IDF.

### 4. Modelo sobre TF-IDF (línea base)

In [ ]:
modelo_tfidf = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
    ("clf", LogisticRegression(max_iter=2000)),
])
t0 = perf_counter()
modelo_tfidf.fit(X_train, y_train)
t_tfidf = perf_counter() - t0
auc_tfidf = roc_auc_score(y_test, modelo_tfidf.predict_proba(X_test)[:, 1])
print(f"TF-IDF completo  -> columnas: {n_cols_tfidf:5d} | tiempo: {t_tfidf:.3f}s | AUC: {auc_tfidf:.3f}")

### 5. TF-IDF + TruncatedSVD a 50 componentes

In [ ]:
modelo_svd = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
    ("svd", TruncatedSVD(n_components=50, random_state=0)),   # funciona sobre sparse
    ("clf", LogisticRegression(max_iter=2000)),
])
t0 = perf_counter()
modelo_svd.fit(X_train, y_train)
t_svd = perf_counter() - t0
auc_svd = roc_auc_score(y_test, modelo_svd.predict_proba(X_test)[:, 1])
print(f"TF-IDF + SVD(50) -> columnas: {50:5d} | tiempo: {t_svd:.3f}s | AUC: {auc_svd:.3f}")

### 6. Comparativa

In [ ]:
tabla = pd.DataFrame({
    "representación": ["TF-IDF completo", "TF-IDF + SVD(50)"],
    "n_columnas": [n_cols_tfidf, 50],
    "tiempo_fit_s": [round(t_tfidf, 3), round(t_svd, 3)],
    "AUC_test": [round(auc_tfidf, 3), round(auc_svd, 3)],
})
print(tabla.to_string(index=False))

### ¿Qué hemos visto?

Pasamos de **miles de columnas dispersas** a solo **50** densas conservando un AUC muy
parecido. Esas 50 componentes son como "temas" que resumen el vocabulario.

Sobre el **tiempo**: aquí el modelo es una regresión logística, que ya es rapidísima sobre una
matriz dispersa, así que añadir el SVD no acelera este paso concreto (incluso puede tardar un
poco más). La ganancia de reducir está en otro sitio: una representación **mucho más compacta
y densa**, ideal para **visualizar, agrupar (clustering) o alimentar modelos más pesados**, y
mucho más barata de almacenar y mover.

Detalle clave: usamos **TruncatedSVD** y no PCA porque trabaja **directamente sobre la matriz
dispersa** (no la centra), que es justo lo que necesita el texto vectorizado.

### Reflexión

1. ¿Cuántas columnas generó TF-IDF y por qué la matriz es tan dispersa?
2. ¿Por qué usamos TruncatedSVD en vez de PCA con datos sparse?
3. Si el AUC apenas cambia, ¿en qué casos merece la pena pasar de miles de columnas a 50?
4. ¿Qué representan, intuitivamente, las 50 componentes de TruncatedSVD?